# TTM Zero-Shot Error Metric Analysis (Single-Pollutant Files)

Computes **RMSE, MAE, MASE** from the saved outputs in `ttm_results_v1/`.

Each subfolder contains results for **one site × one pollutant** (single channel).

- Predictions and actuals are **inverse-scaled** before computing metrics.
- For each site the error is the **mean over the 12 prediction steps**, then reported per rolling window.
- MASE uses a naïve one-step-ahead baseline computed on the context (past) values.
- Mean/Median baselines predict the context mean/median for all 12 prediction steps.
- Outputs: **18 per-window DataFrames** (6 pollutants × 3 methods) + **1 summary DataFrame** averaged over all sites per pollutant.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import pickle as pkl
from tqdm import tqdm

RESULTS_DIR = "/mnt/2TB_WD/rishi/cpcb/ttm_results_v1"
CONTEXT_LENGTH = 168
PREDICTION_LENGTH = 12
POLLUTANTS = ["CO", "NO2", "Ozone", "PM10", "PM2.5", "SO2"]

In [ ]:
def rmse_pw(a, p):
    """(N,) RMSE per window, averaged over horizon."""
    return np.sqrt(np.mean((a - p) ** 2, axis=1))

def mae_pw(a, p):
    """(N,) MAE per window, averaged over horizon."""
    return np.mean(np.abs(a - p), axis=1)

def mase_pw(a, p, past, eps=1e-12):
    """(N,) MASE per window using naïve 1-step-ahead baseline on context."""
    naive_mae = np.mean(np.abs(np.diff(past, axis=1)), axis=1)  # (N,)
    mae_vals = np.mean(np.abs(a - p), axis=1)                   # (N,)
    with np.errstate(invalid="ignore"):
        out = mae_vals / naive_mae
    out[naive_mae < eps] = np.nan
    return out

## Compute Metrics for TTM, Mean Baseline, and Median Baseline

Iterates over all site folders grouped by pollutant. For each folder, computes per-window RMSE, MAE, MASE for:
1. **TTM** predictions
2. **Mean baseline** — predicts the mean of all 168 context values
3. **Median baseline** — predicts the median of all 168 context values

In [ ]:
all_dirs = sorted(
    d for d in os.listdir(RESULTS_DIR)
    if os.path.isdir(os.path.join(RESULTS_DIR, d))
)

# Group folders by pollutant
dirs_by_poll = {p: [] for p in POLLUTANTS}
for d in all_dirs:
    poll = d.rsplit("_", 1)[1]
    site = d.rsplit("_", 1)[0]
    if poll in dirs_by_poll:
        dirs_by_poll[poll].append((site, d))

# Dict to hold 18 per-window DataFrames: key = (pollutant, method)
per_window_dfs = {}

METRICS = ["RMSE", "MAE", "MASE"]

# For each prediction step j, the 7 same-hour-of-day context indices are j, j+24, ..., j+144
# (context is 168h = 7 days, pred step j corresponds to hour j+1 after context end)
same_hour_idx = np.array([[j + 24 * k for k in range(7)] for j in range(PREDICTION_LENGTH)])  # (12, 7)

for poll in POLLUTANTS:
    ttm_rows, mean_rows, median_rows = [], [], []

    for site_name, folder in tqdm(dirs_by_poll[poll], desc=f"Processing {poll}"):
        site_path = os.path.join(RESULTS_DIR, folder)

        # ── Load artefacts ──────────────────────────────────────
        dataset = torch.load(os.path.join(site_path, "dataset.pt"), weights_only=False)
        preds_tensor = torch.load(os.path.join(site_path, "predictions.pt"), weights_only=False)
        with open(os.path.join(site_path, "scaler_params.pkl"), "rb") as f:
            scaler = pkl.load(f)

        future_vals = dataset["future_values"].numpy()[:, :, 0]  # (N, 12)
        past_vals   = dataset["past_values"].numpy()[:, :, 0]    # (N, 168)
        preds_np    = preds_tensor.numpy()[:, :, 0]              # (N, 12)

        mean_  = scaler["mean_"][0]
        scale_ = scaler["scale_"][0]

        # ── Inverse-scale ───────────────────────────────────────
        a = future_vals * scale_ + mean_   # actuals  (N, 12)
        p = preds_np * scale_ + mean_      # TTM preds (N, 12)
        c = past_vals * scale_ + mean_     # context  (N, 168)

        N = a.shape[0]

        # ── Mean & median baselines (N, 12) ─────────────────────
        # Use same-hour-of-day values from context (7 matching hours per step)
        c_same_hour = c[:, same_hour_idx]          # (N, 12, 7)
        mean_bl   = np.mean(c_same_hour, axis=2)   # (N, 12)
        median_bl = np.median(c_same_hour, axis=2) # (N, 12)

        # ── Shared metadata ─────────────────────────────────────
        meta = {"site": [site_name] * N, "window_idx": np.arange(N)}

        # ── TTM metrics ─────────────────────────────────────────
        ttm_rows.append(pd.DataFrame({
            **meta,
            "RMSE": rmse_pw(a, p),
            "MAE":  mae_pw(a, p),
            "MASE": mase_pw(a, p, c),
        }))

        # ── Mean baseline metrics ───────────────────────────────
        mean_rows.append(pd.DataFrame({
            **meta,
            "RMSE": rmse_pw(a, mean_bl),
            "MAE":  mae_pw(a, mean_bl),
            "MASE": mase_pw(a, mean_bl, c),
        }))

        # ── Median baseline metrics ─────────────────────────────
        median_rows.append(pd.DataFrame({
            **meta,
            "RMSE": rmse_pw(a, median_bl),
            "MAE":  mae_pw(a, median_bl),
            "MASE": mase_pw(a, median_bl, c),
        }))

    per_window_dfs[(poll, "TTM")]             = pd.concat(ttm_rows, ignore_index=True)
    per_window_dfs[(poll, "Mean Baseline")]    = pd.concat(mean_rows, ignore_index=True)
    per_window_dfs[(poll, "Median Baseline")]  = pd.concat(median_rows, ignore_index=True)

    print(f"  {poll}: {len(per_window_dfs[(poll, 'TTM')])} windows across {len(dirs_by_poll[poll])} sites")

print(f"\nTotal per-window DataFrames: {len(per_window_dfs)}")

## Per-Pollutant Summary (Averaged Over All Sites)

One summary DataFrame with mean RMSE, MAE, MASE per pollutant per method.

In [4]:
summary_rows = []
for (poll, method), df in per_window_dfs.items():
    summary_rows.append({
        "pollutant": poll,
        "method": method,
        "RMSE_mean":   df["RMSE"].mean(),
        "RMSE_median": df["RMSE"].median(),
        "MAE_mean":    df["MAE"].mean(),
        "MAE_median":  df["MAE"].median(),
        "MASE_mean":   df["MASE"].mean(),
        "MASE_median": df["MASE"].median(),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["pollutant", "method"]).reset_index(drop=True).round(4)
summary_df

,pollutant,method,RMSE_mean,RMSE_median,MAE_mean,MAE_median,MASE_mean,MASE_median
0,CO,Mean Baseline,0.394600,0.253400,0.335100,0.215000,2.817500,1.9649
1,CO,Median Baseline,0.387800,0.235800,0.320500,0.196300,2.753100,1.7943
2,CO,TTM,0.405700,0.264000,0.342900,0.222200,18.642401,2.0134
3,NO2,Mean Baseline,9.623900,5.022400,8.312900,4.311600,3.559700,1.8390
4,NO2,Median Baseline,9.471700,4.662100,7.996600,3.933300,3.517500,1.6311
5,NO2,TTM,9.869600,5.238800,8.478000,4.456300,3.633200,1.8866
6,Ozone,Mean Baseline,16.059601,11.216200,14.143200,9.759100,2.906100,2.3142
7,Ozone,Median Baseline,16.106199,10.148700,13.525400,8.460800,2.814000,2.0191
8,Ozone,TTM,16.428600,11.619600,14.353600,9.999300,2.957500,2.3564
9,PM10,Mean Baseline,52.013699,36.363998,44.566299,31.228800,2.598100,1.9870


## Export

In [ ]:
OUT_DIR = "/home/student/rishi/ttm_results_analysis"
os.makedirs(OUT_DIR, exist_ok=True)

# Export 18 per-window CSVs
for (poll, method), df in per_window_dfs.items():
    safe_poll = poll.replace(".", "").replace(" ", "_")
    safe_method = method.replace(" ", "_").lower()
    fname = f"{safe_poll}_{safe_method}_per_window.csv"
    df.to_csv(os.path.join(OUT_DIR, fname), index=False)

# Export summary
summary_df.to_csv(os.path.join(OUT_DIR, "pollutant_summary.csv"), index=False)

print(f"Exported 18 per-window CSVs + 1 summary CSV to {OUT_DIR}")